In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from pathlib import Path

# ============================================================
# Configuration
# ============================================================
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "legend.fontsize": 10.5,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "axes.linewidth": 1.0,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight"
})

DATA_DIR = Path(".")   # change if needed
OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

files = {
    "biodiversity_total": DATA_DIR / "biodiversity_total.csv",
    "biodiversity_micro": DATA_DIR / "biodiversity_micro.csv",
    "microplastics_yearly": DATA_DIR / "microplastics_yearly.csv",
    "macroplastics_accumulation": DATA_DIR / "macroplastics_accumulation.csv",
}

# ============================================================
# Helpers
# ============================================================
def load_csv(path):
    df = pd.read_csv(path)
    df = df.rename(columns={df.columns[0]: "Category"})
    year_cols = [c for c in df.columns if str(c).isdigit()]
    df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")
    return df, [int(c) for c in year_cols]

def sci_fmt(x, pos):
    if x == 0:
        return "0"
    return f"{x:.1e}"

def clean_label(s):
    return (
        s.replace(" due to ", "\n")
         .replace(" yearly", "\n(yearly)")
         .replace("Total ", "Total\n")
    )

def plot_stacked_with_total(
    df,
    years,
    component_labels,
    total_label,
    title,
    ylabel,
    save_name,
    colors,
    annotate_last=True
):
    years_str = [str(y) for y in years]

    # Extract values
    comp_values = []
    for label in component_labels:
        row = df.loc[df["Category"] == label, years_str]
        if row.empty:
            raise ValueError(f"Could not find row: {label}")
        comp_values.append(row.iloc[0].values.astype(float))

    total_row = df.loc[df["Category"] == total_label, years_str]
    if total_row.empty:
        raise ValueError(f"Could not find total row: {total_label}")
    total_values = total_row.iloc[0].values.astype(float)

    fig, ax = plt.subplots(figsize=(10.5, 6.3))

    # Stacked area
    ax.stackplot(
        years,
        comp_values,
        labels=[clean_label(x) for x in component_labels],
        colors=colors,
        alpha=0.82,
        edgecolor="white",
        linewidth=1.0
    )

    # Total line
    ax.plot(
        years,
        total_values,
        color="black",
        linewidth=2.7,
        marker="o",
        markersize=5,
        label=clean_label(total_label),
        zorder=5
    )

    # Optional annotation on final point
    if annotate_last:
        ax.annotate(
            f"{total_values[-1]:.2e}",
            xy=(years[-1], total_values[-1]),
            xytext=(8, 8),
            textcoords="offset points",
            fontsize=10,
            color="black"
        )

    ax.set_title(title, pad=14, weight="bold")
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_xlim(min(years), max(years))
    ax.yaxis.set_major_formatter(FuncFormatter(sci_fmt))

    ax.grid(True, linestyle="--", alpha=0.28)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    leg = ax.legend(
        loc="upper left",
        frameon=True,
        ncol=1,
        facecolor="white",
        framealpha=0.95
    )
    leg.get_frame().set_linewidth(0.8)

    plt.tight_layout()
    plt.savefig(OUT_DIR / save_name, dpi=300)
    plt.show()
    plt.close()

# ============================================================
# 1) Biodiversity loss: macro vs micro + total
# ============================================================
df1, years1 = load_csv(files["biodiversity_total"])
plot_stacked_with_total(
    df=df1,
    years=years1,
    component_labels=[
        "Biodiversity loss due to macroplastics",
        "Biodiversity loss due to microplastics"
    ],
    total_label="Total biodiversity loss due to macroplastics",
    title="Total Biodiversity Loss from Plastic Pollution",
    ylabel="Biodiversity loss",
    save_name="figure_1_biodiversity_total.png",
    colors=["#4C78A8", "#F58518"]
)

# ============================================================
# 2) Microplastic biodiversity loss: marine vs terrestrial + total
# ============================================================
df2, years2 = load_csv(files["biodiversity_micro"])
plot_stacked_with_total(
    df=df2,
    years=years2,
    component_labels=[
        "Marine biodiversity loss due to microplastics",
        "Terrestrial biodiversity loss due to microplastics"
    ],
    total_label="Total biodiversity loss due to microplastics",
    title="Biodiversity Loss from Microplastics by Receiving Environment",
    ylabel="Biodiversity loss",
    save_name="figure_2_biodiversity_micro.png",
    colors=["#2A9D8F", "#A3B18A"]
)

# ============================================================
# 3) Microplastics yearly: primary vs secondary + total
# ============================================================
df3, years3 = load_csv(files["microplastics_yearly"])
plot_stacked_with_total(
    df=df3,
    years=years3,
    component_labels=[
        "Secondary microplastics yearly",
        "Primary microplastics yearly"
    ],
    total_label="Total microplastics yearly",
    title="Annual Microplastics Generation",
    ylabel="Mass / year",
    save_name="figure_3_microplastics_yearly.png",
    colors=["#6A4C93", "#FFCA3A"]
)

# ============================================================
# 4) Macroplastics accumulation: terrestrial vs marine + total
# ============================================================
df4, years4 = load_csv(files["macroplastics_accumulation"])
plot_stacked_with_total(
    df=df4,
    years=years4,
    component_labels=[
        "Terrestrial macroplastics",
        "Marine macroplastics"
    ],
    total_label="Total macroplastics",
    title="Accumulated Macroplastics by Receiving Environment",
    ylabel="Accumulated mass",
    save_name="figure_4_macroplastics_accumulation.png",
    colors=["#8D6E63", "#1D6996"]
)